In [83]:
import os
from dotenv import load_dotenv
from datasets import load_dataset, Dataset
from langchain_text_splitters import RecursiveCharacterTextSplitter
from huggingface_hub import login
from openai import OpenAI
import json

In [2]:
load_dotenv(override=True)

openai_key = os.getenv('OPENAI_API_KEY')
if openai_key:
    print(f"OpenAi api key exists and starts with {openai_key[:8]}")
else:
    print('OpenAi api key does not available')

OpenAi api key exists and starts with sk-proj-


In [8]:
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Token has not been saved to git credential helper.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-authenticate when pushing to the Hugging Face Hub.
Run the following command in your terminal in case you want to set the 'store' credential helper as default.

git config --global credential.helper store

Read https://git-scm.com/book/en/v2/Git-Tools-Credential-Storage for more details.


In [81]:
dataset = load_dataset("rag-datasets/rag-mini-wikipedia", "text-corpus", split="passages")

if dataset:
    print(f"Total Number of text-corpus: {len(dataset)}")
else:
    print('Issue occured while loading the datasets')

Total Number of text-corpus: 3200


In [24]:
print(dataset[10])

{'passage': 'The inhabitants of Uruguay before European colonization of the area were various tribes of hunter gatherer native Americans, the most well known being the CharrÃºa Indians, a small tribe driven south by the GuaranÃ\xad Indians of Paraguay. The population is estimated at no more than 5000 to 10000.  /ref>', 'id': 10}


In [25]:
def find_longest_passage(dataset, text_field="passage"):
    max_index = -1
    max_text = ""
    max_chars = 0

    for index, row in enumerate(dataset):
        text = row.get(text_field, "")
        if not isinstance(text, str):
            text = str(text)

        char_count = len(text)
        if char_count > max_chars:
            max_index = index
            max_text = text
            max_chars = char_count

    return {
        "index": max_index,
        "char_count": max_chars,
        "passage": max_text,
    }


longest_passage = find_longest_passage(dataset)
print(f"Longest passage index: {longest_passage['index']}")
print(f"Character count: {longest_passage['char_count']}")

Longest passage index: 2095
Character count: 2515


In [27]:
SYSTEM_PROMPT="""
You are a text-cleaning and passage-refinement assistant for a RAG pipeline.

Your task is to improve raw passages so they are easier to read and more useful for semantic retrieval, while preserving the original meaning and factual content.

Rules:
1. Correct grammar, spelling, punctuation, and obvious OCR or formatting issues.
2. Remove unwanted special characters, broken symbols, repeated punctuation, stray markup, and noisy artifacts.
3. Normalize whitespace and line breaks.
4. Rewrite awkward or fragmented sentences into clear, natural, readable English.
5. Preserve key facts, entities, names, numbers, dates, technical terms, and domain-specific keywords.
6. Do not add new facts, opinions, explanations, or assumptions that are not present in the source.
7. Do not summarize unless the passage is extremely repetitive or malformed.
8. Keep the output faithful to the source, but make it cleaner, more coherent, and retrieval-friendly.
9. If the passage contains lists or structured facts, preserve that structure when useful.
10. If text is severely corrupted, produce the cleanest possible faithful reconstruction without inventing missing content.

Output requirements:
- Return only the cleaned passage.
- Do not include labels, commentary, notes, or explanations.
- Do not wrap the result in quotes or markdown.
- Keep the passage in a single clean readable form suitable for embedding and retrieval.
"""

In [39]:
messages=[
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": dataset[2095]["passage"]}
]

In [42]:
MODEL="gpt-5-nano"
openai=OpenAI()
response=openai.chat.completions.create(messages=messages, model=MODEL, reasoning_effort='medium')

prompt_tokens = response.usage.prompt_tokens
completion_tokens = response.usage.completion_tokens

prompt_details = getattr(response.usage, "prompt_tokens_details", None)
cached_tokens = getattr(prompt_details, "cached_tokens", 0) if prompt_details else 0
input_tokens = prompt_tokens - cached_tokens

input_cost = (input_tokens / 1_000_000) * 0.20
cached_input_cost = (cached_tokens / 1_000_000) * 0.02
output_cost = (completion_tokens / 1_000_000) * 1.25

total_cost_usd = input_cost + cached_input_cost + output_cost

print(response.choices[0].message.content)
print()
print(f"Input Tokens: {prompt_tokens}")
print(f"Cached Input Tokens: {cached_tokens}")
print(f"Output Tokens: {completion_tokens}")
print(f"Cost: {total_cost_usd * 100:.3f} cents")


As Ford approached his ninetieth year, he began to experience significant health problems associated with old age. He suffered two minor strokes at the 2000 Republican National Convention, but recovered. BBC reported Ford's recovery on August 2, 2000. In January 2006, he spent 11 days at the Eisenhower Medical Center near his Rancho Mirage, California residence, for treatment of pneumonia. Former President Ford, 92, was hospitalized with pneumonia, according to the Associated Press report on January 17, 2006. On April 23, President George W. Bush visited Ford at his Rancho Mirage home for a little over an hour. This was Ford's last public appearance and produced the last known public photos, video footage, and voice recording. While vacationing in Vail, Colorado, he was hospitalized for two days in July 2006 for shortness of breath. Gerald Ford was released from the hospital. Associated Press, July 26, 2006. On August 15 Ford was admitted to St. Mary’s Hospital of the Mayo Clinic in Ro

In [43]:
MODEL="gpt-5.4-nano-2026-03-17"
openai=OpenAI()
response=openai.chat.completions.create(messages=messages, model=MODEL, reasoning_effort='medium')

prompt_tokens = response.usage.prompt_tokens
completion_tokens = response.usage.completion_tokens

prompt_details = getattr(response.usage, "prompt_tokens_details", None)
cached_tokens = getattr(prompt_details, "cached_tokens", 0) if prompt_details else 0
input_tokens = prompt_tokens - cached_tokens

input_cost = (input_tokens / 1_000_000) * 0.20
cached_input_cost = (cached_tokens / 1_000_000) * 0.02
output_cost = (completion_tokens / 1_000_000) * 1.25

total_cost_usd = input_cost + cached_input_cost + output_cost

print(response.choices[0].message.content)
print()
print(f"Input Tokens: {prompt_tokens}")
print(f"Cached Input Tokens: {cached_tokens}")
print(f"Output Tokens: {completion_tokens}")
print(f"Cost: {total_cost_usd * 100:.3f} cents")


As Ford approached his ninetieth year, he began to experience significant health problems associated with old age. He suffered two minor strokes at the 2000 Republican National Convention but made a quick recovery. Gerald Ford recovering after strokes. BBC, August 2, 2000. Retrieved on December 31, 2006.

In January 2006, he spent 11 days at the Eisenhower Medical Center near his residence in Rancho Mirage, California, for treatment of pneumonia. Former President Ford, 92, hospitalized with pneumonia. Associated Press, January 17, 2006. Retrieved on October 19, 2007.

On April 23, President George W. Bush visited Ford at his home in Rancho Mirage for a little over an hour. This was Ford’s last public appearance and produced the last known public photos, video footage, and voice recording. While vacationing in Vail, Colorado, he was hospitalized for two days in July 2006 for shortness of breath. Gerald Ford released from hospital. Associated Press, July 26, 2006. Retrieved on December 3

In [52]:
def make_jsonl(item):
    body = {"model": "gpt-5.4-nano-2026-03-17", "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": item["passage"]},
        ], "reasoning_effort":'medium'}
    line = {"custom_id": str(item["id"]), "method": "POST", "url": "/v1/chat/completions", "body": body}
    return json.dumps(line)

In [47]:
print(make_jsonl(dataset[2095]))

{"custom_id": "2096", "method": "POST", "url": "/v1/chat/completions", "body": {"model": "gpt-5.4-nano-2026-03-17", "messages": [{"role": "system", "content": "\nYou are a text-cleaning and passage-refinement assistant for a RAG pipeline.\n\nYour task is to improve raw passages so they are easier to read and more useful for semantic retrieval, while preserving the original meaning and factual content.\n\nRules:\n1. Correct grammar, spelling, punctuation, and obvious OCR or formatting issues.\n2. Remove unwanted special characters, broken symbols, repeated punctuation, stray markup, and noisy artifacts.\n3. Normalize whitespace and line breaks.\n4. Rewrite awkward or fragmented sentences into clear, natural, readable English.\n5. Preserve key facts, entities, names, numbers, dates, technical terms, and domain-specific keywords.\n6. Do not add new facts, opinions, explanations, or assumptions that are not present in the source.\n7. Do not summarize unless the passage is extremely repetit

In [53]:
def make_file(start, end, filename):
    os.makedirs(os.path.dirname(filename), exist_ok=True)

    with open(filename, "w", encoding="utf-8") as f:
        for i in range(start, end):
            f.write(make_jsonl(dataset[i]))
            f.write("\n")

In [56]:
make_file(2001, 3200, "jsonl/2001_3200.jsonl")

In [69]:
batch_input=openai.files.create(
    file=open("jsonl/2001_3200.jsonl", "rb"),
    purpose="batch"
)

print(batch_input)

FileObject(id='file-VpQdYyCTJukSpmAvyFeiVy', bytes=2486529, created_at=1779273155, filename='2001_3200.jsonl', object='file', purpose='batch', status='processed', expires_at=1781865155, status_details=None)


In [70]:
batch_input_id = batch_input.id
openai.batches.create(
    input_file_id=batch_input_id,
    endpoint="/v1/chat/completions",
    completion_window="24h",
    metadata={
    "dataset": "rag-mini-wikipedia",
    "config": "text-corpus",
    "split": "passages",
    "row_range": "1001-2000",
    "task": "passage_refinement",
    "model": "gpt-5.4-nano-2026-03-17",
    "prompt_version": "v1"
}
)

Batch(id='batch_6a0d8dc7c68c81909d97939033d605b6', completion_window='24h', created_at=1779273159, endpoint='/v1/chat/completions', input_file_id='file-VpQdYyCTJukSpmAvyFeiVy', object='batch', status='validating', cancelled_at=None, cancelling_at=None, completed_at=None, error_file_id=None, errors=None, expired_at=None, expires_at=1779359559, failed_at=None, finalizing_at=None, in_progress_at=None, metadata={'dataset': 'rag-mini-wikipedia', 'config': 'text-corpus', 'split': 'passages', 'row_range': '1001-2000', 'task': 'passage_refinement', 'model': 'gpt-5.4-nano-2026-03-17', 'prompt_version': 'v1'}, model=None, output_file_id=None, request_counts=BatchRequestCounts(completed=0, failed=0, total=0), usage=BatchUsage(input_tokens=0, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=0, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=0))

In [76]:
batch = openai.batches.retrieve("batch_6a0d8d97b18c8190a2e86f8c4d528526")
print(batch.status)
print(batch.request_counts)
print(batch.output_file_id)
print(batch.error_file_id)

completed
BatchRequestCounts(completed=999, failed=0, total=999)
file-FKZyn6mwuCoqC5rG9yBXpc
None


In [66]:
file_response=openai.files.content("file-4us3L5qC7zV1b2XcpnYMn8")
print(file_response.text)

{"id": "batch_req_6a0d87f2643c8190995ef5a9d268773e", "custom_id": "0", "response": {"status_code": 200, "request_id": "df22a4cb-8659-46ff-97fc-6f43b967cb06", "body": {"id": "chatcmpl-DhY5vlXd9o28uBCVJ6R5gInjWupyy", "object": "chat.completion", "created": 1779271535, "model": "gpt-5.4-nano-2026-03-17", "choices": [{"index": 0, "message": {"role": "assistant", "content": "Uruguay (official full name in ; pronunciation ; Eastern Republic of Uruguay) is a country located in the southeastern part of South America. It is home to 3.3 million people, of which 1.7 million live in the capital, Montevideo, and its metropolitan area.", "refusal": null, "annotations": []}, "finish_reason": "stop"}], "usage": {"prompt_tokens": 347, "completion_tokens": 61, "total_tokens": 408, "prompt_tokens_details": {"cached_tokens": 0, "audio_tokens": 0}, "completion_tokens_details": {"reasoning_tokens": 0, "audio_tokens": 0, "accepted_prediction_tokens": 0, "rejected_prediction_tokens": 0}}, "service_tier": "def

In [79]:
OUTPUT_FILE_IDS = [
    "file-4us3L5qC7zV1b2XcpnYMn8",
    "file-FKZyn6mwuCoqC5rG9yBXpc",
    "file-47XVZUHYMs1gHqvNdGXoMV",
]

def extract_message_content(message_content):
    if isinstance(message_content, str):
        return message_content

    if isinstance(message_content, list):
        parts = []
        for item in message_content:
            if isinstance(item, dict) and item.get("type") == "text":
                parts.append(item.get("text", ""))
        return "\n".join(part for part in parts if part).strip()

    return str(message_content)

def load_batch_outputs(file_ids, client):
    corrected_by_custom_id = {}

    for file_id in file_ids:
        file_text = client.files.content(file_id).text
        for line in file_text.splitlines():
            if not line.strip():
                continue

            record = json.loads(line)

            custom_id = str(record["custom_id"])
            error = record.get("error")
            if error is not None:
                print(f"Skipping failed record: {custom_id} -> {error}")
                continue

            body = record["response"]["body"]
            content = body["choices"][0]["message"]["content"]
            corrected_passage = extract_message_content(content)

            corrected_by_custom_id[custom_id] = corrected_passage

    return corrected_by_custom_id

corrected_map = load_batch_outputs(OUTPUT_FILE_IDS, openai)

print(f"Corrected passages loaded: {len(corrected_map)}")
print(corrected_map.get("2095"))

Corrected passages loaded: 3198
In a prerecorded, embargoed interview with Bob Woodward of The Washington Post in July 2004, Ford stated that he disagreed “very strongly” with the Bush administration’s choice of Iraq’s alleged weapons of mass destruction as justification for its decision to invade Iraq. He called it a “big mistake” unrelated to the national security of the United States and indicated that he would not have gone to war had he been President. The details of the interview were not released until after Ford’s death, as he requested.  

Woodward, Bob. (December 28, 2006). “Ford Disagreed With Bush About Invading Iraq.” The Washington Post. Retrieved on December 28, 2006.  

Democracy Now Headlines for December 28, 2006. “Embargoed Interview Reveals Ford Opposed Iraq War.” Retrieved on December 28, 2006.


In [86]:
def build_refined_dataset(original_dataset, corrected_map):
    refined_rows = []
    missing_ids = []

    for row in original_dataset:
        row_id = str(row["id"])

        if row_id not in corrected_map:
            missing_ids.append(row_id)
            continue

        new_row = dict(row)
        new_row["original_passage"] = row["passage"]
        new_row["passage"] = corrected_map[row_id]
        refined_rows.append(new_row)

    return Dataset.from_list(refined_rows), missing_ids

refined_dataset, missing_ids = build_refined_dataset(dataset, corrected_map)

print(refined_dataset[0]["id"],refined_dataset[0]["passage"] )
print(f"Missing ids: {len(missing_ids)}")
if missing_ids:
    print(missing_ids[:10])

0 Uruguay (official full name in ; pronunciation ; Eastern Republic of Uruguay) is a country located in the southeastern part of South America. It is home to 3.3 million people, of which 1.7 million live in the capital, Montevideo, and its metropolitan area.
Missing ids: 2
['1001', '2001']


In [87]:
print(refined_dataset.column_names)
print(refined_dataset.features)

['passage', 'id', 'original_passage']
{'passage': Value('string'), 'id': Value('int64'), 'original_passage': Value('string')}


In [89]:
publish_dataset = refined_dataset.select_columns(["id", "passage", "original_passage"])

print(publish_dataset.column_names)
print(publish_dataset[0])

['id', 'passage', 'original_passage']
{'id': 0, 'passage': 'Uruguay (official full name in ; pronunciation ; Eastern Republic of Uruguay) is a country located in the southeastern part of South America. It is home to 3.3 million people, of which 1.7 million live in the capital, Montevideo, and its metropolitan area.', 'original_passage': 'Uruguay (official full name in  ; pron.  , Eastern Republic of  Uruguay) is a country located in the southeastern part of South America.  It is home to 3.3 million people, of which 1.7 million live in the capital Montevideo and its metropolitan area.'}


In [90]:
repo_id = "Pratheep17/rag-mini-wikipedia-refined"

publish_dataset.push_to_hub(
    repo_id,
    config_name="text-corpus",
    split="passages",
    private=False
)

Setting num_proc from 1 back to 1 for the passages split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 49.30ba/s]
Processing Files (1 / 1): 100%|██████████| 1.55MB / 1.55MB,  104kB/s  
New Data Upload: 100%|██████████| 1.55MB / 1.55MB,  104kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:10<00:00, 10.31s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/Pratheep17/rag-mini-wikipedia-refined/commit/b365b1f8b2e8b00cfb0da1c278ef032221fce442', commit_message='Upload dataset', commit_description='', oid='b365b1f8b2e8b00cfb0da1c278ef032221fce442', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Pratheep17/rag-mini-wikipedia-refined', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Pratheep17/rag-mini-wikipedia-refined'), pr_revision=None, pr_num=None)